# Apps Migration: Reconciliation

This notebook generates a reconciliation report comparing source and target state.

## Configuration

In [ ]:
try:
    target_catalog = dbutils.widgets.get("target_catalog")
except:
    target_catalog = "YOUR_TARGET_CATALOG"

try:
    target_schema = dbutils.widgets.get("target_schema")
except:
    target_schema = "apps_demo"

try:
    import_volume = dbutils.widgets.get("import_volume")
except:
    import_volume = "apps_migration"

try:
    target_parent_path = dbutils.widgets.get("target_parent_path")
except:
    target_parent_path = "/Workspace/Shared/MigratedApps"

volume_path = f"/Volumes/{target_catalog}/{target_schema}/{import_volume}"
print(f"Volume path: {volume_path}")

## Load results

In [ ]:
from databricks.sdk import WorkspaceClient
import json
from io import BytesIO, StringIO
import csv

w = WorkspaceClient()

deployment_results = []
permissions_results = {}

try:
    response = w.files.download(f"{volume_path}/deployment_results.json")
    deployment_results = json.loads(response.contents.read().decode('utf-8'))
    print(f"Loaded {len(deployment_results)} deployment results")
except Exception as e:
    print(f"Warning: {e}")

try:
    response = w.files.download(f"{volume_path}/permissions_results.json")
    permissions_results = json.loads(response.contents.read().decode('utf-8'))
    print(f"Loaded permissions results for {len(permissions_results)} apps")
except Exception as e:
    print(f"Warning: {e}")

## Generate reconciliation report

In [ ]:
entries = []

for result in deployment_results:
    app_name = result.get("target_name", "")
    
    app_status = "N/A"
    if result.get("status") in ["CREATED", "UPDATED"]:
        try:
            app = w.apps.get(name=app_name)
            if app.active_deployment and app.active_deployment.status:
                app_status = str(app.active_deployment.status.value)
            else:
                app_status = "PENDING"
        except:
            app_status = "ERROR"
    
    entries.append({
        "source_name": result.get("source_name", ""),
        "target_name": app_name,
        "target_url": result.get("target_url", ""),
        "deployment_status": result.get("status", ""),
        "app_status": app_status,
        "notes": result.get("message", ""),
    })

print(f"Generated {len(entries)} reconciliation entries")

## Write report

In [ ]:
report_path = f"{volume_path}/reconciliation_report.csv"

output = StringIO()
writer = csv.DictWriter(output, fieldnames=["source_name", "target_name", "target_url", "deployment_status", "app_status", "notes"])
writer.writeheader()
writer.writerows(entries)

w.files.upload(report_path, BytesIO(output.getvalue().encode('utf-8')), overwrite=True)
print(f"Reconciliation report: {report_path}")

## Summary

In [ ]:
import pandas as pd

total = len(entries)
created = len([e for e in entries if e['deployment_status'] == "CREATED"])
updated = len([e for e in entries if e['deployment_status'] == "UPDATED"])
failed = len([e for e in entries if e['deployment_status'] == "FAILED"])

print("=" * 60)
print("RECONCILIATION COMPLETE")
print("=" * 60)
print(f"Total apps: {total}")
print(f"  Created: {created}")
print(f"  Updated: {updated}")
print(f"  Failed: {failed}")
print("=" * 60)

if entries:
    display(pd.DataFrame(entries))

# Fail the job if there are deployment failures
if failed > 0:
    failed_apps = [e['source_name'] for e in entries if e['deployment_status'] == "FAILED"]
    raise Exception(
        f"Migration failed for {failed} app(s): {', '.join(failed_apps)}. "
        f"Check deployment_results.json and reconciliation_report.csv for details."
    )